# Slide-level classification · slide encoder

Predict a slide-level label using a **slide-native encoder** — a model that
runs its own tile encoder internally and returns **one vector per slide**:

```
Dataset -> FeatureExtractor (slide encoder) -> train (head, no aggregator) -> evaluate
```

There is no bag to pool, so **no MIL aggregator** (`aggregator=None`). The
alternative — a tile encoder whose per-tile bag is pooled by an aggregator —
is the [tile-encoder + MIL walkthrough](walkthrough-slide-mil.ipynb), and the
two differ by exactly the encoder name and that one argument.

> Tiny synthetic data, CPU-only, ungated encoder — the numbers are
> meaningless; the point is the API.

## ⚠️ Scaffolding (not soma API)

The cell below fabricates toy slides and the two CSVs soma expects. **Replace
this with your own slides and labels** — only the on-disk contract matters:

* `dataset.csv` — one row per slide: `sample_id`, `image_path`, `label`.
* `splits.csv` — `sample_id`, `split` (`train` / `tune` / `test*`), optional
  `fold`.

In [ ]:
import logging, warnings
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

WORK = Path(tempfile.mkdtemp(prefix='soma-slide-encoder-'))
SLIDES = WORK / 'slides'; SLIDES.mkdir()
rng = np.random.default_rng(0)

def make_toy_slide(path, size=640):
    """A white background with a central H&E-ish blob, saved as a tiled TIFF
    whose resolution tags make OpenSlide report 0.5 microns/pixel."""
    img = np.full((size, size, 3), 240, np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]
    blob = ((xx - size // 2) ** 2 + (yy - size // 2) ** 2) < (size * 0.35) ** 2
    tissue = np.stack([np.full((size, size), 150),
                       np.full((size, size), 70),
                       np.full((size, size), 160)], -1).astype(np.int16)
    tissue += rng.integers(-30, 30, (size, size, 3))
    img[blob] = np.clip(tissue, 0, 255).astype(np.uint8)[blob]
    tifffile.imwrite(path, img, photometric='rgb', tile=(256, 256),
                     resolution=(20000, 20000), resolutionunit='CENTIMETER')

N = 8
sample_ids = [f's{i:02d}' for i in range(N)]
for sid in sample_ids:
    make_toy_slide(SLIDES / f'{sid}.tif')

# train/tune/test assignment (single fold) with both classes in every split
split = (['train'] * 4) + (['tune'] * 2) + (['test'] * 2)
binary = [0, 1, 0, 1,  0, 1,  0, 1]

dataset_csv = WORK / 'dataset.csv'
splits_csv = WORK / 'splits.csv'
pd.DataFrame({'sample_id': sample_ids,
              'image_path': [str(SLIDES / f'{s}.tif') for s in sample_ids],
              'label': binary}).to_csv(dataset_csv, index=False)
pd.DataFrame({'sample_id': sample_ids, 'split': split}).to_csv(splits_csv, index=False)

print(pd.read_csv(dataset_csv).head().to_string(index=False))

## 1. Load the dataset and splits

Identical to every other path — `Dataset` reads the manifest and infers the
label space; `Splits` pairs the provided splits with it, unchanged.

In [ ]:
from soma import Dataset, Splits

dataset = Dataset(dataset_csv)
splits = Splits(splits_csv, dataset)
print('slides:', len(dataset.sample_ids), '| folds:', splits.num_folds)

## 2. Extract one vector per slide

A **slide-level** encoder runs its own tile encoder internally and returns a
single vector per slide, so the store holds one vector each — not a bag. We
use [moozy-slide](https://huggingface.co/AtlasAnalyticsLab/MOOZY) (built on
the ungated `lunit` tile encoder) so this runs on CPU with no token; swap in
`titan`, `prism`, or `gigapath-slide` if you have access.

In [ ]:
from soma import FeatureExtractor, EncoderConfig, PreprocessingConfig, CacheConfig

extractor = FeatureExtractor(
    dataset,
    EncoderConfig(name='moozy-slide', allow_non_recommended_settings=True),
    preprocessing=PreprocessingConfig(
        backend='openslide', requested_tile_size_px=224, requested_spacing_um=0.5,
        tissue_method='otsu', seg_downsample=16, a_t=1,
    ),
    cache=CacheConfig(enabled=True, root_dir=str(WORK / 'cache')),
    output_root=str(WORK / 'output'),
)
store = extractor.extract(feature_dir='features')
vec = store.load(store.available_samples[0])
print('slide-level features:', store.is_slide_level, '| one vector of dim', tuple(vec.shape))

## 3. Train the head — no aggregator

The store already holds one vector per slide, so `train()` attaches the task
head directly with `aggregator=None`. Everything else — task heads, metrics,
and swapping to multiclass/regression/survival on the same store — is exactly
as in the [MIL walkthrough](walkthrough-slide-mil.ipynb); only the aggregator
is gone.

In [ ]:
from soma import TaskConfig, TrainingConfig, EvalConfig, train

result = train(
    feature_store=store,
    dataset=dataset,
    splits=splits,
    aggregator=None,          # <- slide-level features need no aggregator
    task=TaskConfig(name='binary_classification'),
    training=TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4, seed=0),
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    run_dir=str(WORK / 'runs' / 'binary'),
)
print('run dir:', result.run_dir)

## 4. The one-shot `Pipeline` equivalent

The same run as a single config-driven call. Note the absent `aggregator` —
the only structural difference from the MIL pipeline config.

*(Shown for reference, not executed.)*

```python
from soma import (
    Pipeline, PipelineConfig, PreprocessingConfig, EncoderConfig,
    TaskConfig, TrainingConfig, EvalConfig, CacheConfig,
)

config = PipelineConfig(
    dataset_csv=str(dataset_csv),
    splits_csv=str(splits_csv),
    output_root='output/binary',
    dataset_type='slide',
    preprocessing=PreprocessingConfig(
        backend='openslide', requested_tile_size_px=224, requested_spacing_um=0.5,
        tissue_method='otsu',
    ),
    encoder=EncoderConfig(name='moozy-slide', allow_non_recommended_settings=True),
    aggregator=None,          # <- slide encoder: no bag to pool
    task=TaskConfig(name='binary_classification'),
    training=TrainingConfig(epochs=3, learning_rate=1e-3, batch_size=4),
    evaluation=EvalConfig(metrics=['balanced_accuracy', 'auroc']),
    cache=CacheConfig(enabled=True),
)
results = Pipeline(config).run()
```